# ☁️ Trading Forecasting - Deploy no SageMaker

Este notebook implementa o deploy dos modelos treinados no Amazon SageMaker Pipeline.

## Objetivos:
- Configurar pipeline SageMaker
- Fazer upload dos dados para S3
- Criar e executar pipeline de ML
- Monitorar execução
- Realizar inferência na nuvem

In [ ]:
# Importações necessárias
import boto3
import sagemaker
import pandas as pd
import os
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# SageMaker imports
from sagemaker.sklearn.processing import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput
from sagemaker.workflow.parameters import ParameterString, ParameterInteger
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.steps import ProcessingStep
from sagemaker.workflow.pipeline_context import PipelineSession

# Configurações AWS
session = sagemaker.Session()
region = session.boto_region_name
role = sagemaker.get_execution_role()
default_bucket = session.default_bucket()

print(f"✅ Configuração AWS:")
print(f"   Região: {region}")
print(f"   Bucket padrão: {default_bucket}")
print(f"   Role: {role}")

## 1. Preparação dos Dados para S3

In [ ]:
def prepare_and_upload_data():
    """Prepara e faz upload dos dados para S3"""
    
    print("📦 Preparando dados para upload...")
    
    # Verificar se existem dados locais
    data_files = {
        'dados_preparados': '../data/dados_preparados.csv',
        'historico_propostas': '../data/historico_propostas.xlsx'
    }
    
    uploaded_files = {}
    
    for data_name, local_path in data_files.items():
        if os.path.exists(local_path):
            # Upload para S3
            s3_key = f"trading-forecasting/data/{data_name}_{datetime.now().strftime('%Y%m%d')}.{local_path.split('.')[-1]}"
            s3_uri = f"s3://{default_bucket}/{s3_key}"
            
            # Upload usando SageMaker session
            uploaded_uri = session.upload_data(
                path=local_path,
                bucket=default_bucket,
                key_prefix="trading-forecasting/data"
            )
            
            uploaded_files[data_name] = uploaded_uri
            print(f"   ✅ {data_name} -> {uploaded_uri}")
        else:
            print(f"   ⚠️ Arquivo não encontrado: {local_path}")
    
    # Se não há dados locais, criar dados de exemplo
    if not uploaded_files:
        print("   📋 Criando dados de exemplo...")
        
        # Criar dados sintéticos
        import numpy as np
        np.random.seed(42)
        
        df_exemplo = pd.DataFrame({
            'CodigoMaterial': np.random.choice(['10000', '10001', '10002', '10003'], 1000),
            'Fornecedor': np.random.choice(['Alpha Ltda', 'Beta Corp', 'Gamma SA', 'Delta Inc'], 1000),
            'Quantidade': np.random.randint(50, 500, 1000),
            'PrazoEntrega(dias)': np.random.randint(5, 30, 1000),
            'CondicoesPagamento': np.random.choice(['À vista', '30 dias', '60 dias'], 1000),
            'FreteIncluso': np.random.choice(['Sim', 'Não'], 1000),
            'ValidadeProposta(dias)': np.random.randint(15, 60, 1000),
            'PrecoUnitario(R$)': np.random.uniform(100, 500, 1000),
            'DescontoAplicavel(%)': np.random.uniform(0, 25, 1000)
        })
        
        # Salvar temporariamente
        temp_file = '/tmp/dados_exemplo.csv'
        df_exemplo.to_csv(temp_file, index=False)
        
        # Upload para S3
        uploaded_uri = session.upload_data(
            path=temp_file,
            bucket=default_bucket,
            key_prefix="trading-forecasting/data"
        )
        
        uploaded_files['dados_exemplo'] = uploaded_uri
        print(f"   ✅ dados_exemplo -> {uploaded_uri}")
        
        # Limpeza
        os.remove(temp_file)
    
    return uploaded_files

# Executar upload
uploaded_data = prepare_and_upload_data()
print(f"\n📊 Total de arquivos no S3: {len(uploaded_data)}")

## 2. Preparação dos Scripts de Treinamento e Inferência

In [ ]:
def upload_processing_scripts():
    """Faz upload dos scripts de processamento para S3"""
    
    print("📝 Preparando scripts de processamento...")
    
    scripts_uploaded = {}
    
    # Lista de scripts necessários
    scripts = {
        'sagemaker_train.py': '../scripts/sagemaker_train.py',
        'sagemaker_inference.py': '../scripts/sagemaker_inference.py'
    }
    
    for script_name, script_path in scripts.items():
        if os.path.exists(script_path):
            # Upload script para S3
            uploaded_uri = session.upload_data(
                path=script_path,
                bucket=default_bucket,
                key_prefix="trading-forecasting/scripts"
            )
            
            scripts_uploaded[script_name] = uploaded_uri
            print(f"   ✅ {script_name} -> {uploaded_uri}")
        else:
            print(f"   ⚠️ Script não encontrado: {script_path}")
    
    return scripts_uploaded

# Upload dos scripts
uploaded_scripts = upload_processing_scripts()
print(f"\n🔧 Scripts carregados: {len(uploaded_scripts)}")

## 3. Criação do Pipeline SageMaker

In [ ]:
def create_sagemaker_pipeline(input_data_uri):
    """Cria pipeline SageMaker para treinamento e inferência"""
    
    print("🏗️ Criando pipeline SageMaker...")
    
    # Pipeline session
    pipeline_session = PipelineSession()
    
    # Parâmetros do pipeline
    input_data_param = ParameterString(
        name="InputDataUri",
        default_value=input_data_uri
    )
    
    model_bucket_param = ParameterString(
        name="ModelBucket",
        default_value=default_bucket
    )
    
    # Processador para treinamento
    training_processor = SKLearnProcessor(
        framework_version="0.23-1",
        instance_type="ml.m5.xlarge",
        instance_count=1,
        role=role,
        sagemaker_session=pipeline_session
    )
    
    # Step 1: Treinamento
    training_step = ProcessingStep(
        name="TrainingModels",
        processor=training_processor,
        inputs=[
            ProcessingInput(
                source=input_data_param,
                destination="/opt/ml/processing/input"
            )
        ],
        outputs=[
            ProcessingOutput(
                output_name="models",
                source="/opt/ml/processing/output/models",
                destination=f"s3://{model_bucket_param}/trading-forecasting/models/"
            ),
            ProcessingOutput(
                output_name="metrics",
                source="/opt/ml/processing/output/metrics",
                destination=f"s3://{model_bucket_param}/trading-forecasting/metrics/"
            )
        ],
        code="../scripts/sagemaker_train.py"  # Caminho local do script
    )
    
    # Processador para inferência
    inference_processor = SKLearnProcessor(
        framework_version="0.23-1",
        instance_type="ml.m5.large",
        instance_count=1,
        role=role,
        sagemaker_session=pipeline_session
    )
    
    # Step 2: Inferência
    inference_step = ProcessingStep(
        name="InferenceStep",
        processor=inference_processor,
        inputs=[
            ProcessingInput(
                source=training_step.properties.ProcessingOutputConfig.Outputs["models"].S3Output.S3Uri,
                destination="/opt/ml/processing/input/models"
            )
        ],
        outputs=[
            ProcessingOutput(
                output_name="predictions",
                source="/opt/ml/processing/output/predictions",
                destination=f"s3://{model_bucket_param}/trading-forecasting/predictions/"
            )
        ],
        code="../scripts/sagemaker_inference.py"  # Caminho local do script
    )
    
    # Criar pipeline
    pipeline = Pipeline(
        name="TradingForecastingPipeline",
        parameters=[input_data_param, model_bucket_param],
        steps=[training_step, inference_step],
        sagemaker_session=pipeline_session
    )
    
    print(f"   ✅ Pipeline criado: {pipeline.name}")
    print(f"   📊 Steps: {len(pipeline.steps)}")
    
    return pipeline

# Criar pipeline se temos dados
if uploaded_data:
    # Usar o primeiro arquivo de dados disponível
    data_uri = list(uploaded_data.values())[0]
    print(f"📁 Usando dados: {data_uri}")
    
    pipeline = create_sagemaker_pipeline(data_uri)
else:
    print("❌ Nenhum dado disponível para criar pipeline")
    pipeline = None

## 4. Deploy e Execução do Pipeline

In [ ]:
def deploy_and_execute_pipeline(pipeline):
    """Faz deploy e executa o pipeline"""
    
    if pipeline is None:
        print("❌ Pipeline não disponível para deploy")
        return None
    
    try:
        print("🚀 Fazendo deploy do pipeline...")
        
        # Deploy do pipeline
        pipeline.upsert(role_arn=role)
        print(f"   ✅ Pipeline '{pipeline.name}' deployed com sucesso!")
        
        # Executar pipeline
        print("\n▶️ Iniciando execução do pipeline...")
        execution = pipeline.start()
        
        print(f"   🎯 Execution ARN: {execution.arn}")
        print(f"   📊 Status: {execution.describe()['PipelineExecutionStatus']}")
        
        # Informações para monitoramento
        print(f"\n📋 Para monitorar a execução:")
        print(f"   - SageMaker Console: https://{region}.console.aws.amazon.com/sagemaker/home?region={region}#/pipelines")
        print(f"   - AWS CLI: aws sagemaker describe-pipeline-execution --pipeline-execution-arn {execution.arn}")
        
        return execution
        
    except Exception as e:
        print(f"❌ Erro durante deploy/execução: {str(e)}")
        print(f"\n💡 Possíveis soluções:")
        print(f"   - Verificar permissões IAM")
        print(f"   - Verificar se os scripts estão corretos")
        print(f"   - Verificar cotas de instâncias")
        return None

# Executar deploy se temos pipeline
if 'pipeline' in locals() and pipeline is not None:
    execution = deploy_and_execute_pipeline(pipeline)
else:
    execution = None
    print("⚠️ Pipeline não disponível - pule para a seção de monitoramento manual")

## 5. Monitoramento da Execução

In [ ]:
def monitor_pipeline_execution(execution):
    """Monitora o status da execução do pipeline"""
    
    if execution is None:
        print("❌ Nenhuma execução para monitorar")
        return
    
    try:
        print("👀 Monitorando execução do pipeline...")
        
        # Obter status atual
        status_info = execution.describe()
        status = status_info['PipelineExecutionStatus']
        
        print(f"\n📊 Status atual: {status}")
        print(f"⏰ Iniciado em: {status_info.get('CreationTime', 'N/A')}")
        
        # Listar steps e seus status
        steps = execution.list_steps()
        print(f"\n📋 Status dos steps:")
        
        for step in steps:
            step_name = step['StepName']
            step_status = step['StepStatus']
            print(f"   {step_name}: {step_status}")
            
            # Se houver erro, mostrar detalhes
            if step_status == 'Failed' and 'FailureReason' in step:
                print(f"      ❌ Erro: {step['FailureReason']}")
        
        # Instruções baseadas no status
        if status == 'Executing':
            print(f"\n⏳ Pipeline em execução. Tempo estimado: 10-20 minutos")
            print(f"💡 Execute esta célula novamente para atualizar o status")
        elif status == 'Succeeded':
            print(f"\n✅ Pipeline executado com sucesso!")
            print(f"📁 Resultados disponíveis em: s3://{default_bucket}/trading-forecasting/")
        elif status == 'Failed':
            print(f"\n❌ Pipeline falhou. Verifique os logs para mais detalhes.")
        elif status == 'Stopping' or status == 'Stopped':
            print(f"\n⏹️ Pipeline foi interrompido.")
        
        return status_info
        
    except Exception as e:
        print(f"❌ Erro ao monitorar pipeline: {str(e)}")
        return None

# Monitorar se temos execução
if 'execution' in locals() and execution is not None:
    status_info = monitor_pipeline_execution(execution)
else:
    print("ℹ️ Para monitorar uma execução existente, forneça o ARN da execução:")
    print("# execution_arn = 'arn:aws:sagemaker:region:account:pipeline-execution/pipeline-name/execution-id'")
    print("# execution = session.sagemaker_client.describe_pipeline_execution(PipelineExecutionArn=execution_arn)")

## 6. Verificação dos Resultados

In [ ]:
def check_pipeline_outputs():
    """Verifica os outputs do pipeline no S3"""
    
    print("📁 Verificando outputs do pipeline...")
    
    s3_client = boto3.client('s3')
    
    # Prefixos para verificar
    prefixes = [
        'trading-forecasting/models/',
        'trading-forecasting/metrics/',
        'trading-forecasting/predictions/'
    ]
    
    all_outputs = {}
    
    for prefix in prefixes:
        try:
            response = s3_client.list_objects_v2(
                Bucket=default_bucket,
                Prefix=prefix
            )
            
            if 'Contents' in response:
                files = [obj['Key'] for obj in response['Contents']]
                all_outputs[prefix] = files
                print(f"\n📂 {prefix}:")
                for file in files[:5]:  # Mostrar apenas os primeiros 5
                    file_size = next(obj['Size'] for obj in response['Contents'] if obj['Key'] == file)
                    print(f"   📄 {file} ({file_size:,} bytes)")
                
                if len(files) > 5:
                    print(f"   ... e mais {len(files) - 5} arquivos")
            else:
                print(f"\n📂 {prefix}: (vazio)")
                all_outputs[prefix] = []
                
        except Exception as e:
            print(f"❌ Erro ao verificar {prefix}: {str(e)}")
            all_outputs[prefix] = []
    
    # Resumo
    total_files = sum(len(files) for files in all_outputs.values())
    print(f"\n📊 Resumo: {total_files} arquivos encontrados")
    
    return all_outputs

# Verificar outputs
pipeline_outputs = check_pipeline_outputs()

# Sugestões baseadas nos outputs encontrados
if any(pipeline_outputs.values()):
    print(f"\n💡 Próximos passos:")
    if pipeline_outputs.get('trading-forecasting/models/'):
        print(f"   ✅ Modelos encontrados - pipeline de treinamento executou com sucesso")
    if pipeline_outputs.get('trading-forecasting/predictions/'):
        print(f"   ✅ Previsões encontradas - pipeline de inferência executou com sucesso")
    print(f"   📥 Faça download dos resultados usando AWS CLI ou console")
    print(f"   🔄 Execute novamente o pipeline com novos dados se necessário")
else:
    print(f"\n⚠️ Nenhum output encontrado. Possíveis causas:")
    print(f"   - Pipeline ainda em execução")
    print(f"   - Pipeline falhou durante execução")
    print(f"   - Problemas de permissão no S3")

## 7. Download de Resultados (Opcional)

In [ ]:
def download_pipeline_results():
    """Faz download dos resultados do pipeline"""
    
    print("📥 Fazendo download dos resultados...")
    
    # Criar diretórios locais
    os.makedirs('../results/models', exist_ok=True)
    os.makedirs('../results/metrics', exist_ok=True)
    os.makedirs('../results/predictions', exist_ok=True)
    
    s3_client = boto3.client('s3')
    downloaded_files = []
    
    # Mapear prefixos S3 para diretórios locais
    download_mapping = {
        'trading-forecasting/models/': '../results/models/',
        'trading-forecasting/metrics/': '../results/metrics/',
        'trading-forecasting/predictions/': '../results/predictions/'
    }
    
    for s3_prefix, local_dir in download_mapping.items():
        if s3_prefix in pipeline_outputs and pipeline_outputs[s3_prefix]:
            print(f"\n📂 Baixando arquivos de {s3_prefix}...")
            
            for s3_key in pipeline_outputs[s3_prefix][:3]:  # Limitar a 3 arquivos por categoria
                try:
                    # Nome do arquivo local
                    filename = os.path.basename(s3_key)
                    local_path = os.path.join(local_dir, filename)
                    
                    # Download
                    s3_client.download_file(default_bucket, s3_key, local_path)
                    downloaded_files.append(local_path)
                    print(f"   ✅ {filename}")
                    
                except Exception as e:
                    print(f"   ❌ Erro ao baixar {s3_key}: {str(e)}")
    
    print(f"\n📊 Total de arquivos baixados: {len(downloaded_files)}")
    
    if downloaded_files:
        print(f"\n📁 Arquivos salvos em: ../results/")
        for file in downloaded_files:
            print(f"   📄 {file}")
    
    return downloaded_files

# Opção de download (descomente para executar)
# downloaded_files = download_pipeline_results()

print("ℹ️ Para fazer download dos resultados, descomente e execute a linha acima")

## 8. Configuração para Execuções Futuras

In [ ]:
# Salvar configurações para uso futuro
deployment_config = {
    'timestamp': datetime.now().isoformat(),
    'pipeline_name': 'TradingForecastingPipeline',
    'region': region,
    'bucket': default_bucket,
    'role': role,
    'uploaded_data': uploaded_data,
    'pipeline_deployed': pipeline is not None,
    'execution_arn': execution.arn if execution else None
}

# Salvar configuração
config_file = '../sagemaker_deployment_config.json'
with open(config_file, 'w') as f:
    json.dump(deployment_config, f, indent=2)

print(f"💾 Configuração salva em: {config_file}")

# Comandos úteis para o futuro
print(f"\n🔧 COMANDOS ÚTEIS PARA ADMINISTRAÇÃO:")
print(f"\n📋 Listar execuções do pipeline:")
print(f"aws sagemaker list-pipeline-executions --pipeline-name TradingForecastingPipeline")

print(f"\n🗑️ Deletar pipeline (se necessário):")
print(f"aws sagemaker delete-pipeline --pipeline-name TradingForecastingPipeline")

print(f"\n📁 Listar arquivos no S3:")
print(f"aws s3 ls s3://{default_bucket}/trading-forecasting/ --recursive")

print(f"\n💰 Monitorar custos:")
print(f"aws ce get-cost-and-usage --time-period Start=2024-01-01,End=2024-12-31 --granularity MONTHLY --metrics BlendedCost --group-by Type=DIMENSION,Key=SERVICE")

print(f"\n✅ Deploy concluído! Pipeline está pronto para uso.")
print(f"\n🎯 Para executar novamente:")
print(f"   1. Carregue novos dados para S3")
print(f"   2. Execute: pipeline.start()")
print(f"   3. Monitore via SageMaker Console")